In [5]:
!pip -q install transformers gdown scikit-learn pandas

In [6]:
!gdown --folder "https://drive.google.com/drive/folders/1WbTqPWLGTGkJFgVxVehPMjM0rYGUwwG7" -O dataset/xsum-translation
!find dataset -name "*test*"

Retrieving folder contents
Processing file 1rnRgAk2AgeHtSe4tx4DOqqbcEstAlXf5 en-translated-test.csv
Processing file 1hrDkOfmnAPQynYVpoQ3zYSJbc5hrhggh en-translated-train.csv
Processing file 1DCOFqudYQ0KsgzT0LUEXOkDkGZz2jYzi en-translated-valid.csv
Processing file 17k9egUTFktnJeXq8M3xFw_LmpHdppabG es-translated-test.csv
Processing file 1qqOq3xxGeVNe9jgVRTw9AI8Vwekb6pyH es-translated-train.csv
Processing file 13Rt-w5z5qeYCdReInkYke-jkl27VpkXQ es-translated-valid.csv
Processing file 1K4RjpoS77tn-PrUf4ccdhavQWAlD8elE fr-translated-test.csv
Processing file 17szR-_vOtd_9DKFczMx3pGIRmAKAiL4l fr-translated-train.csv
Processing file 1-Y1xaZAM1yYdoWL1EKoy-yVI3_EuOVjc fr-translated-valid.csv
Processing file 1-E-6sL-fijwarWBIb93ybeOauYij825V hi-translated-test.csv
Processing file 1Cfk0iZoWJsLGvExVx9WnMotDOMEGnUk8 hi-translated-train.csv
Processing file 1jCJ8oSMjLYp1X8RouwVqCPij9GNEHWaq hi-translated-valid.csv
Processing file 1LL7L4EIrdEvSOKvtTjdL7uKoSBqzTvWE tr-translated-test.csv
Processing file 

In [7]:
import glob, pandas as pd
f = glob.glob("dataset/**/es-translated*test*.csv", recursive=True)[0]
df = pd.read_csv(f)
print(f, len(df)); print(df.columns.tolist()); df.head(2)

dataset/xsum-translation/es-translated-test.csv 500
['premise', 'hypothesis', 'label']


,premise,hypothesis,label
0,Millard se unió a Cardiff Blues en 2015 despué...,Cardiff Blues ha confirmado que el gerente gen...,contradiction
1,"El equipo de la Universidad Sheffield Hallam, ...",Los investigadores del Reino Unido están lleva...,entailment


In [8]:
import torch, glob, numpy as np, pandas as pd, os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

DOC, SUM, LAB = "document", "summary", "label"   # <-- EDIT to match Cell 3
dev = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("results/table10", exist_ok=True)

def run_lang(lang, model_id=None, bs=16):
    path = glob.glob(f"dataset/**/{lang}*test*.csv", recursive=True)[0]
    d = pd.read_csv(path)
    mid = model_id or f"yfqiu-nlp/mFACT-{lang}"
    tok = AutoTokenizer.from_pretrained(mid)
    model = AutoModelForSequenceClassification.from_pretrained(mid).to(dev).eval()
    probs = []
    for i in range(0, len(d), bs):
        b = d.iloc[i:i+bs]
        enc = tok(list(b[DOC].astype(str)), list(b[SUM].astype(str)),
                  truncation="only_first", max_length=512,
                  padding=True, return_tensors="pt").to(dev)
        with torch.no_grad():
            probs += model(**enc).logits.softmax(-1)[:, 1].cpu().tolist()
    d["p_faithful"] = probs
    d["pred"] = (d["p_faithful"] > 0.5).astype(int)
    d.to_csv(f"results/table10/{lang}_preds.csv", index=False)   # raw outputs for the repo
    y = d[LAB].values
    p, r, f1, _ = precision_recall_fscore_support(y, d["pred"], average="binary", pos_label=1)
    return dict(lang=lang, n=len(d), acc=100*accuracy_score(y, d["pred"]),
                prec=100*p, rec=100*r, f1=100*f1)

In [11]:
import glob, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Correct the column mapping variables in the namespace
globals()["DOC"] = "premise"
globals()["SUM"] = "hypothesis"
globals()["LAB"] = "label"

# Map paper keys to the corresponding filename prefixes
lang_file_map = {
    "zh_CN": "zh",
    "es_XX": "es",
    "fr_XX": "fr",
    "hi_IN": "hi",
    "vi_VN": "vi",
    "tr_TR": "tr"
}

# Define a robust version of run_lang that resolves the file paths correctly
def run_lang_fixed(lang, model_id=None, bs=16):
    lang_prefix = lang_file_map.get(lang, lang.split('_')[0])
    path = glob.glob(f"dataset/**/{lang_prefix}-translated*test*.csv", recursive=True)[0]
    d = pd.read_csv(path)
    mid = model_id or f"yfqiu-nlp/mFACT-{lang}"
    tok = AutoTokenizer.from_pretrained(mid)
    model = AutoModelForSequenceClassification.from_pretrained(mid).to(dev).eval()
    probs = []
    for i in range(0, len(d), bs):
        b = d.iloc[i:i+bs]
        enc = tok(list(b[DOC].astype(str)), list(b[SUM].astype(str)),
                  truncation="only_first", max_length=512,
                  padding=True, return_tensors="pt").to(dev)
        with torch.no_grad():
            probs += model(**enc).logits.softmax(-1)[:, 1].cpu().tolist()
    d["p_faithful"] = probs
    d["pred"] = (d["p_faithful"] > 0.5).astype(int)
    d.to_csv(f"results/table10/{lang}_preds.csv", index=False)
    y_true = (d[LAB] == "entailment").astype(int).values
    p, r, f1, _ = precision_recall_fscore_support(y_true, d["pred"], average="binary", pos_label=1)
    return dict(lang=lang, n=len(d), acc=100*accuracy_score(y_true, d["pred"]),
                prec=100*p, rec=100*r, f1=100*f1)

paper = {  # Table 10, mFACT rows: acc, prec, rec, f1
 "zh_CN":(92.60,92.71,92.34,92.53), "es_XX":(97.00,97.55,96.37,96.96),
 "fr_XX":(95.60,95.20,95.97,95.58), "hi_IN":(94.00,95.04,92.74,93.88),
 "vi_VN":(95.40,94.47,96.37,95.41), "tr_TR":(97.00,97.17,96.77,96.97)}

rows = []
for lang in paper:
    r = run_lang_fixed(lang); rows.append(r)
    print(r)
res = pd.DataFrame(rows).set_index("lang")
for i, m in enumerate(["acc","prec","rec","f1"]):
    res[m+"_paper"] = [paper[l][i] for l in res.index]
    res[m+"_delta"] = res[m] - res[m+"_paper"]
res.round(2).to_csv("results/table10/table10_comparison.csv")
res.round(2)

config.json:   0%|          | 0.00/900 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/360 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  711MB            

... (57 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  711MB            

model.safetensors: downloading bytes:           |  0.00B            

{'lang': 'tr_TR', 'n': 500, 'acc': 97.0, 'prec': 97.16599190283401, 'rec': 96.7741935483871, 'f1': 96.96969696969697}


,n,acc,prec,rec,f1,acc_paper,acc_delta,prec_paper,prec_delta,rec_paper,rec_delta,f1_paper,f1_delta
lang,,,,,,,,,,,,,
zh_CN,500,95.2,96.28,93.95,95.10,92.6,2.6,92.71,3.57,92.34,1.61,92.53,2.57
es_XX,500,97.0,97.55,96.37,96.96,97.0,0.0,97.55,0.00,96.37,0.00,96.96,-0.00
fr_XX,500,95.6,95.20,95.97,95.58,95.6,0.0,95.20,-0.00,95.97,-0.00,95.58,0.00
hi_IN,500,94.0,95.04,92.74,93.88,94.0,0.0,95.04,0.00,92.74,0.00,93.88,-0.00
vi_VN,500,95.4,94.47,96.37,95.41,95.4,-0.0,94.47,-0.00,96.37,0.00,95.41,-0.00
tr_TR,500,97.0,97.17,96.77,96.97,97.0,0.0,97.17,-0.00,96.77,0.00,96.97,-0.00


In [12]:
t1 = pd.DataFrame({"mean": res[["acc","prec","rec","f1"]].mean(),
                   "sd": res[["acc","prec","rec","f1"]].std()})   # sample SD
t1["paper_mean"] = [95.3, 95.4, 95.1, 95.2]
t1["paper_sd"]   = [1.7, 1.8, 2.0, 1.8]
t1.round(2).to_csv("results/table10/table1_comparison.csv")
t1.round(2)

,mean,sd,paper_mean,paper_sd
acc,95.70,1.15,95.3,1.7
prec,95.95,1.24,95.4,1.8
rec,95.36,1.63,95.1,2.0
f1,95.65,1.18,95.2,1.8


In [13]:
import os, glob, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
dev = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("results", exist_ok=True)
LANGS = {"zh_CN":"zh","es_XX":"es","fr_XX":"fr","hi_IN":"hi","vi_VN":"vi","tr_TR":"tr"}

def load_model(lang):
    mid = f"yfqiu-nlp/mFACT-{lang}"
    return (AutoTokenizer.from_pretrained(mid),
            AutoModelForSequenceClassification.from_pretrained(mid).to(dev).eval())

def score(tok, model, docs, sums, max_len=512, bs=16):
    out = []
    for i in range(0, len(docs), bs):
        enc = tok(list(docs[i:i+bs]), list(sums[i:i+bs]), truncation="only_first",
                  max_length=max_len, padding=True, return_tensors="pt").to(dev)
        with torch.no_grad():
            out += model(**enc).logits.softmax(-1)[:, 1].cpu().tolist()
    return np.array(out)

In [14]:
rows = []
for lang, pre in LANGS.items():
    d = pd.read_csv(glob.glob(f"dataset/**/{pre}-translated*test*.csv", recursive=True)[0])
    docs, sums = d["premise"].astype(str).values, d["hypothesis"].astype(str).values
    y = (d["label"] == "entailment").astype(int).values
    tok, model = load_model(lang)
    conds = {"original": docs,
             "shuffled_docs": np.roll(docs, 137),       # each summary gets another article
             "summary_only": np.array([""] * len(docs))}
    for name, dd in conds.items():
        pred = (score(tok, model, dd, sums) > 0.5).astype(int)
        rows.append(dict(lang=lang, condition=name, acc=100*(pred == y).mean(),
                         pct_pred_faithful=100*pred.mean(),
                         on_gold_faithful=100*pred[y==1].mean(),
                         on_gold_halluc=100*pred[y==0].mean()))
    del model; torch.cuda.empty_cache()
probe = pd.DataFrame(rows).round(1)
probe.to_csv("results/probe_document_use.csv", index=False)
probe.pivot(index="lang", columns="condition", values="acc")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

condition,original,shuffled_docs,summary_only
lang,,,
es_XX,97.0,52.2,50.4
fr_XX,95.6,52.6,50.4
hi_IN,94.0,57.8,50.4
tr_TR,97.0,53.4,50.4
vi_VN,95.4,54.6,50.4
zh_CN,95.2,54.2,53.0


In [15]:
from sklearn.metrics import f1_score
ts = np.round(np.arange(0.1, 0.95, 0.1), 1)
sweep = {}
for lang in LANGS:
    d = pd.read_csv(f"results/table10/{lang}_preds.csv")
    y = (d["label"] == "entailment").astype(int)
    sweep[lang] = [100*f1_score(y, (d["p_faithful"] > t).astype(int)) for t in ts]
sweep = pd.DataFrame(sweep, index=ts).round(2)
sweep.to_csv("results/threshold_sweep_f1.csv"); sweep

,zh_CN,es_XX,fr_XX,hi_IN,vi_VN,tr_TR
0.1,95.81,96.19,94.86,91.98,94.47,96.21
0.2,95.97,96.58,95.03,93.89,95.22,96.19
0.3,95.55,96.76,95.39,94.19,95.22,96.39
0.4,95.12,96.96,95.39,93.69,95.41,96.77
0.5,95.10,96.96,95.58,93.88,95.41,96.97
0.6,95.10,96.96,95.58,93.85,95.58,97.35
0.7,95.49,96.96,95.58,94.05,95.97,97.35
0.8,95.04,97.14,95.58,94.05,95.76,96.51
0.9,93.72,96.72,96.15,92.95,95.55,96.08
